In [2]:
import numpy as np

def viewfac_tree(H, W, rttt, dt, ht):
    #------------------------------------------------------------------------
    #  Purpose:
    #     compute view factors for street canyons with shade trees
    #
    #  Synopsis:
    #     FGS,FWW,FGW,FWG,FWS = viewfac_tree(H, W, rttt, dt, ht)
    #
    #  Variable Description:
    #     H  - normalized building height
    #     W  - normalized road width
    #     rttt - tree ring radius
    #     dt - distance of tree to nearest wall
    #     ht - height of tree crown center
    #------------------------------------------------------------------------

    NMC = 10000  # number of samples in MC simulation
    zc = ht  # vertical location of center of tree crown
    xc = dt  # horizontal location of the center of tree crown
    nF = np.zeros((5, 5))  # bundle number matrix

    for i in range(NMC):
        Reta = np.random.rand()
        Rtheta = np.random.rand()
        theta = 2 * np.pi * Rtheta
        eta = np.arcsin(np.sqrt(Reta))
        Rx = np.random.rand()
        Rz = np.random.rand()  # random point of emission from canyon facets
        Re = np.random.rand()  # emission from tree
        Xe = W * Rx
        Ze = H * Rz  # physical coordinates
        Xet = xc + rttt * np.cos(2 * np.pi * Re)
        Zet = zc + rttt * np.sin(2 * np.pi * Re)  # tree emission

        # compute coordinates of incident points
        # 1. between canyon facets
        xsg = Xe + H * np.tan(eta) * np.cos(theta)
        xgs = xsg
        zww = Ze + W * np.tan(eta) * np.cos(theta)
        zgw = Xe / np.tan(eta) / np.cos(theta)
        zsw = zgw
        xwg = Ze / np.tan(eta) / np.cos(theta)
        xws = xwg

        # 2. from tree to canyon facets
        dtw = xc - rttt / 2
        dtg = zc - rttt / 2
        dts = H - zc + rttt / 2
        dtt = W - 2 * dtw + rttt
        dtw2 = W - 2 * xc + rttt
        xtg = Xet + dtg * np.tan(eta) * np.cos(theta)
        xts = Xet + dts * np.tan(eta) * np.cos(theta)
        ztw = Zet + dtw * np.tan(eta) * np.cos(theta)
        ztw2 = Zet + dtw2 * np.tan(eta) * np.cos(theta)  # far end wall
        ztt = Zet + dtt * np.tan(eta) * np.cos(theta)

        # 3. from canyon facets to tree
        zwt = Ze + dtw * np.tan(eta) * np.cos(theta)
        xwt = Ze / np.tan(eta) / np.cos(theta)
        xgt = Xe + dtg * np.tan(eta) * np.cos(theta)
        xst = Xe + dts * np.tan(eta) * np.cos(theta)
        zgt = Xe / np.tan(eta) / np.cos(theta)
        zst = zgt

        # view factor from sky to tree/ground FST/FSG
        bst = (zst >= max(0, zc - rttt) and zst <= min(H, zc + rttt)) or \
              (xst >= max(0, xc - rttt) and xst <= min(W, xc + rttt))
        if bst:
            nF[0, 3] += 1
        if not bst and 0 <= xsg <= W:
            nF[0, 1] += 1

        # view factor from ground to tree/sky FGT/FGS
        bgt = (zgt >= max(0, zc - rttt) and zgt <= min(H, zc + rttt)) or \
              (xgt >= max(0, xc - rttt) and xgt <= min(W, xc + rttt))
        if bgt:
            nF[1, 3] += 1
        if not bgt and 0 <= xgs <= W:
            nF[1, 0] += 1

        # view factor from sky/ground to wall FSW/FGW
        if not bst and 0 <= zsw <= H:
            nF[0, 2] += 1
        if not bgt and 0 <= zgw <= H:
            nF[1, 2] += 1

        # view factor from wall to tree/wall
        bwt = (zwt >= max(0, zc - rttt) and zwt <= min(H, zc + rttt)) or \
              (xwt >= max(0, xc - rttt) and xwt <= min(W, xc + rttt))
        if bwt:
            nF[2, 3] += 1
        if not bwt and 0 <= zww <= H:
            nF[2, 2] += 1

        # view factors from wall to sky/ground
        if not bwt and 0 <= xws <= W:
            nF[2, 0] += 1
        if not bwt and 0 <= xwg <= W:
            nF[2, 1] += 1

        # view factor from tree to wall/ground/sky
        if 0 <= xtg <= W:
            nF[3, 1] += 1
        if 0 <= xts <= W:
            nF[3, 0] += 1
        if 0 <= ztw <= H:
            nF[3, 2] += 1
        # far end wall
        if 0 <= ztw2 <= H:
            nF[4, 4] += 1

        # view factor between trees
        if H - zc - rttt <= ztt <= H - zc + rttt:
            nF[3, 3] += 1

    vF = nF / NMC
    vF[3, :] = vF[3, :] / 2  # accounts for the symmetry of emission from trees
    vF[4, 4] = vF[4, 4] / 2
    vF[3, 2] = (vF[3, 2] + vF[4, 4]) / 2

    FGS = vF[1, 0]
    FWW = vF[2, 2]
    FGW = vF[1, 2]
    FWG = vF[2, 1]
    FWS = vF[2, 0]

    return FGS, FWW, FGW, FWG, FWS


In [3]:
# Testing the function with example values
H, W, rttt, dt, ht = 0.2, 0.5, 2, 2, 5     #adjust it based on your own data
FGS, FWW, FGW, FWG, FWS = viewfac_tree(H, W, rttt, dt, ht)
FGS, FWW, FGW, FWG, FWS

(0.6171, 0.1492, 0.156, 0.0, 0.0)